# Optional after-class extension: learn to hop

Select a GPU runtime and run setup before enabling training.
This extension is separate from the completed walking lesson.

A hop lifts **both feet** before a controlled landing. This recorded hopper
reached about 5.9 cm bilateral clearance and 0.16 s of clear flight; its feet
still tilt about 9°. Foot-marker height alone does not prove obstacle clearance.

| Earlier vs updated hopping | Same motion at half speed |
|:---:|:---:|
| ![Recorded hopping comparison](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/hopping.gif?v=a07c95656f35) | ![Recorded hopping comparison at half speed](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/inline/hopping_slow.gif?v=4c47173d257a) |

Watch both feet leave the ground, then inspect the landing.

### A curriculum adds one skill at a time

**Rhythm → clearance → foot control → landing.** The goal is not to make every
row taller. Some stages return to flat ground to isolate a new skill before
adding obstacles again. Each stage starts from its predecessor's networks.

| Phase | Stages, in order | What changes—and why |
|---|---|---|
| **1 · Find a rhythm** | 1. Tiny hops → 2. Moving hops | Start on flat ground; then add speed and turning commands. |
| **2 · Clear the ground** | 3. Terrain hops | Measure the whole sole in flight and introduce 25 mm terrain. |
| **3 · Control both feet** | 4. Level hops → 5. Level terrain hops | Reward simultaneous clearance and flatter soles on flat ground; then restore terrain. |
| **4 · Refine the landing** | 6. Natural hops → 7. Block hops → 8. Soft landing hops | Level feet during flight only, practise steps, then reduce body-lift reward. |

The last phases keep the same 6.5 cm foot target while changing **how** the
robot takes off, clears terrain, and lands. The eight recorded stages took
about **23 minutes** on an L40S, excluding setup and evaluation.

| Recorded hopping measurements |
|:---:|
| ![Measured instructor hopping](https://raw.githubusercontent.com/TheNewRobot/bdxr-fastsac-classroom/main/assets/lesson/notebook/course_hops-motion.png?v=3a4a6eee5854) |

### Your optional activity · train and validate hopping

Enable the toggle below to generate your own policies and videos. Leaving it
off does not train or download an instructor checkpoint.

In [ ]:
# @title Unpack the lesson files
import hashlib
import runpy
import subprocess
import sys
import tempfile
from pathlib import Path

# Download a small helper and verify it before running any downloaded code.
BUNDLE_URL = "https://github.com/TheNewRobot/bdxr-fastsac-classroom/releases/download/lesson-e9831819255a/BDXR_lesson_files.zip"
BUNDLE_SHA256 = "e9831819255aaab9012954b2d2332a4302b291066bb2a25f98661254e5d2a02b"
BOOTSTRAP_SHA256 = "9492e3ba32bed43816f4bf799bb8f60e7f3ab12fa24f30ff7457ff3a7e51e0fb"
WORK = (
    Path("/content/bdxr_classroom") if Path("/content").exists()
    else Path.cwd() / "bdxr_classroom_lab"
) / BUNDLE_SHA256[:12]
helper = Path(tempfile.mkdtemp(prefix="bdxr_setup_")) / "lesson_download.py"
fetch = "from urllib.request import urlretrieve; import sys; urlretrieve(*sys.argv[1:])"
subprocess.run(
    [sys.executable, "-c", fetch, BUNDLE_URL.rsplit("/", 1)[0] + "/lesson_download.py", str(helper)],
    check=True, timeout=60,
)
if hashlib.sha256(helper.read_bytes()).hexdigest() != BOOTSTRAP_SHA256:
    raise RuntimeError("Setup helper verification failed. Rerun this cell.")
runpy.run_path(str(helper))["prepare_lesson"](BUNDLE_URL, BUNDLE_SHA256, WORK)

In [ ]:
# @title Install pinned dependencies
if "WORK" not in globals() or not (WORK / "scripts/setup/setup.py").is_file():
    raise RuntimeError("Run the unpack cell first.")

from bdxr_classroom.lesson.setup import install_runtime

PYTHON, runtime_env = install_runtime(WORK)

In [ ]:
import json
import os
from pathlib import Path
from bdxr_classroom.lesson.runtime import Lesson
from bdxr_classroom.lesson.hopping import load_curriculum, train_stage

lab = Lesson(WORK, PYTHON, runtime_env)
CURRICULUM = load_curriculum(WORK)
LEARNING_CONFIG = {
    "seed": 7, "num_envs": 512, "batch_size": 2048, "buffer_size": 512,
    "actor_hidden": 256, "critic_hidden": 384, "gamma": 0.97,
    "updates": 4, "compile": True, "wandb_mode": "disabled",
}

In [ ]:
TRAIN_HOPPING = False  # @param {type:"boolean"}
HOP_STAGES = []
if TRAIN_HOPPING:
    previous = None
    for stage in CURRICULUM:
        print("Training stage:", stage["name"])
        previous = train_stage(lab, stage, LEARNING_CONFIG, previous)
        HOP_STAGES.append(previous)

Choose a stage to inspect after training. Later stages are not automatically
better. Look for both feet leaving the ground and a controlled landing;
height of a foot marker alone is not proof of obstacle clearance.
These recordings are observations, not certification of a hopping controller.

In [ ]:
from IPython.display import display, Video

STAGE_TO_WATCH = 6  # @param {type:"integer"}
if HOP_STAGES:
    if not 1 <= STAGE_TO_WATCH <= len(HOP_STAGES):
        raise ValueError("Choose a stage number between 1 and 8.")
    chosen = HOP_STAGES[STAGE_TO_WATCH - 1]
    display(Video(str(lab.record(chosen)), embed=True, width=640))
else:
    print("Enable hopping training above to produce your own policies.")

In [ ]:
from bdxr_classroom.lesson.exports import export_results

if HOP_STAGES:
    print("Saved:", export_results(WORK, lab.runs, include_replay=True))